In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# Audio features for clustering
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

N_CLUSTERS = 10

# Load your data
df = pd.read_csv(r'/Users/andrewbilbrey/Documents/3 - Data/GitHub Updates/dataset.csv')  
df = df.rename(columns={'track_name': 'name', 'artists': 'artists', 'album_name':'album', 'track_id':'track_id'})
print(df.columns.tolist())
print(f"✓ Loaded {len(df)} songs")
print(f"Columns: {df.columns.tolist()}")


['Unnamed: 0', 'track_id', 'artists', 'album', 'name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
✓ Loaded 114000 songs
Columns: ['Unnamed: 0', 'track_id', 'artists', 'album', 'name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


In [2]:
def preprocess_data(df):
    """Preprocess the data by handling missing values."""
    df_clean = df.copy()
    
    # Remove duplicates
    if 'name' in df_clean.columns and 'artists' in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=['name', 'artists'])
    
    # Handle missing values in audio features
    for feature in AUDIO_FEATURES:
        if feature in df_clean.columns:
            df_clean[feature].fillna(df_clean[feature].median(), inplace=True)
    
    # Remove rows with missing audio features
    df_clean = df_clean.dropna(subset=AUDIO_FEATURES)
    
    print(f"After preprocessing: {len(df_clean)} songs")
    return df_clean



    


In [3]:
def create_clusters(df, n_clusters=N_CLUSTERS):
    """Create K-Means clusters based on audio features."""
    feature_data = df[AUDIO_FEATURES].copy()
    
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(feature_data)
    
    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(features_scaled)
    
    print(f"Created {n_clusters} clusters")
    return df, scaler, kmeans



    
    


In [4]:
def find_song(df, song_name, artist_name=None):
    """Find a song in the dataset."""
    song_name_lower = song_name.lower()
    
    if artist_name:
        artist_name_lower = artist_name.lower()
        matches = df[
            (df['name'].str.lower().str.contains(song_name_lower, na=False)) &
            (df['artists'].str.lower().str.contains(artist_name_lower, na=False))
        ]
    else:
        matches = df[df['name'].str.lower().str.contains(song_name_lower, na=False)]
    
    if len(matches) == 0:
        print(f"No songs found matching '{song_name}'")
        return None
    elif len(matches) > 1:
        print(f"\nFound {len(matches)} matches:")
        for idx, row in matches.head(5).iterrows():
            print(f"  - {row['name']} by {row['artists']}")
        return matches.iloc[0]
    else:
        return matches.iloc[0]


In [5]:
def get_recommendations(df, song_name, artist_name=None, n_recommendations=10):
    """Get song recommendations based on K-Means clustering."""
    # Find the input song
    input_song = find_song(df, song_name, artist_name)
    
    if input_song is None:
        return None
    
    print(f"\nInput song: {input_song['name']} by {input_song['artists']}")
    print(f"Cluster: {input_song['cluster']}")

In [6]:
print(df.columns.tolist())
print(df.head()[0:5].T)   # transpose so you can quickly inspect values

['Unnamed: 0', 'track_id', 'artists', 'album', 'name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
                                       0                       1  \
Unnamed: 0                             0                       1   
track_id          5SuOikwiRyPMVoIQDJUgSV  4qPNDBW1i3p13qLCt0Ki3A   
artists                      Gen Hoshino            Ben Woodward   
album                             Comedy        Ghost (Acoustic)   
name                              Comedy        Ghost - Acoustic   
popularity                            73                      55   
duration_ms                       230666                  149610   
explicit                           False                   False   
danceability                       0.676                    0.42   
energy                             0.461               

In [7]:
df.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album', 'name', 'popularity',
       'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre'],
      dtype='object')

In [8]:
df_clean = preprocess_data(df)

After preprocessing: 81344 songs


In [9]:
df_clustered, scaler, kmeans = create_clusters(df_clean, n_clusters=N_CLUSTERS)


Created 10 clusters


In [10]:
recs = get_recommendations(df_clustered, 'Hold On', artist_name=None, n_recommendations=10)
if recs is not None:
    print(recs.to_string(index=False))



Found 46 matches:
  - Hold On by Chord Overstreet
  - Hold On - Remix by Chord Overstreet;Deepend
  - Hold On by KT Tunstall
  - Hold On - Acoustic by Chord Overstreet
  - Hold On by Greensky Bluegrass

Input song: Hold On by Chord Overstreet
Cluster: 2


In [11]:
recs = get_recommendations(df_clustered, 'Teenage Dream', artist_name=None, n_recommendations=10)
if recs is not None:
    print(recs.to_string(index=False))


Found 4 matches:
  - Teenage Dream by The Rescues
  - Teenage Dream by Sara Farell
  - Teenage Dream - Acoustic by Glee Cast
  - Teenage Dream (feat. Darren Criss) by Glee Cast;Darren Criss

Input song: Teenage Dream by The Rescues
Cluster: 7


In [17]:
from scipy.spatial.distance import cdist

def get_recommendations(df, song_name, artist_name=None, n_recommendations=10, id_col='track_id'):
    # find input song row
    input_song = find_song(df, song_name, artist_name)
    if input_song is None:
        return None

    # prepare feature matrix and scaled features (fit already done when clustering)
    feature_matrix = df[AUDIO_FEATURES].values
    # If you saved scaler when creating clusters, use it. Otherwise fit a new one (but keep consistent).
    features_scaled = scaler.transform(feature_matrix)

    # input song features (raw) -> scaled
    input_idx = input_song.name  # row index in df
    input_feat_raw = df.loc[input_idx, AUDIO_FEATURES].values.reshape(1, -1)
    input_feat_scaled = scaler.transform(input_feat_raw)

    # compute distances to all songs
    distances = cdist(input_feat_scaled, features_scaled, metric='euclidean')[0]

    # build a results DataFrame
    results = df.copy()
    results['distance'] = distances

    # exclude the exact same track (by track_id) and optionally duplicates with same name+artist
    results = results[results[id_col] != input_song[id_col]]

    # sort by distance and return top N
    recommendations = results.nsmallest(n_recommendations, 'distance')
    return recommendations[['track_name' if 'track_name' in df.columns else 'name', 'artists', 'album', 'popularity', 'distance']]

# Example usage (after you created df_clustered, scaler, kmeans):
recs = get_recommendations(df_clustered, 'Hold On', artist_name=None, n_recommendations=10)
print(recs.to_string(index=False))



Found 46 matches:
  - Hold On by Chord Overstreet
  - Hold On - Remix by Chord Overstreet;Deepend
  - Hold On by KT Tunstall
  - Hold On - Acoustic by Chord Overstreet
  - Hold On by Greensky Bluegrass
                     name                 artists                    album  popularity  distance
                      Run                   Kyson              Lucky Notes           0  0.362025
                       分裂                Jay Chou                     八度空間          47  0.377778
               Tid tröste              Mando Diao           I solnedgången          29  0.384509
                     我的背影                 Tai Chi                 太極30週年精選          20  0.410892
           Follow The Sun             Xavier Rudd              Spirit Bird          71  0.418666
 Ölüm Aklımı Çelmek Üzere    Şanışer;Sezgin Alkan              LUDOVICO II          42  0.447653
        Incomparável Amor           Gabriel Brito        Incomparável Amor          47  0.453145
                     

NameError: name 'kendrick' is not defined